# Task 4: Content Segmentation & Clustering

Unsupervised grouping of the catalog:
1. Build a feature space from genres, format, rating, release year, and duration.
2. Choose $K$ with the **elbow method** and **silhouette scores**.
3. Fit **K-Means** at the best $K$ and project titles to 2D/3D with **PCA**.
4. Profile the resulting clusters.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data

df = get_preprocessed_data('../data/Dataset.csv')
print(f"Loaded {len(df):,} titles with {len(df.columns)} columns.")

Loaded 8,790 titles with 19 columns.


## 1. Choosing K

In [2]:
from src.clustering import MovieClusterer

metrics = MovieClusterer().compute_elbow_metrics(df, k_range=range(2, 9))
pd.DataFrame({
    'K': metrics['k_values'],
    'Inertia': np.round(metrics['inertias'], 1),
    'Silhouette': metrics['silhouettes'],
})

,K,Inertia,Silhouette
0,2,23313.0,0.3489
1,3,17974.3,0.3652
2,4,16532.0,0.2598
3,5,15603.4,0.2004
4,6,14154.6,0.1816
5,7,13163.9,0.2081
6,8,12518.7,0.2227


## 2. PCA Explained Variance

In [3]:
from src.clustering import build_clusterer

best_k = metrics['k_values'][int(np.argmax(metrics['silhouettes']))]
print(f"Best K by silhouette: {best_k}")
clusterer = build_clusterer(df, n_clusters=best_k)
for i, var in enumerate(clusterer.pca_3d.explained_variance_ratio_, 1):
    print(f"Component {i}: {var:.2%}")

Best K by silhouette: 3


Component 1: 40.30%
Component 2: 21.89%
Component 3: 7.31%


## 3. Cluster Profiles

In [4]:
summary = clusterer.get_cluster_summary()
summary[['cluster_label', 'count', 'dominant_type', 'dominant_rating', 'avg_release_year', 'sample_titles']]

,cluster_label,count,dominant_type,dominant_rating,avg_release_year,sample_titles
0,Group 0: Dramas & Comedies (Movies),5495,Movie,TV-MA,2015.7,"Dick Johnson Is Dead, Confessions of an Invisi..."
1,Group 1: International TV Shows & Crime TV Sho...,2658,TV Show,TV-MA,2017.0,"Ganglands, Midnight Mass, The Great British Ba..."
2,Group 2: Action & Adventure & Dramas (Movies),637,Movie,R,1988.9,"Sankofa, Jeans, Minsara Kanavu"


## 4. Takeaways

- Format and maturity rating dominate the principal components, so clusters largely separate along those lines.
- Cluster profiles give a quick map of where the catalog is dense and where it is thin.